In [1]:
import numpy as np
import torch
import torch.utils.data as data
import torch.nn as nn
import sys
import warnings
from itertools import product

## NoteBook environment specific (has to be configured on user side)

In [2]:
# import github repo into notebook file directory, might require fixing path
!rm -r /Architectural-Biases-in-Time-Series-Anomaly-Detection
!git clone https://github.com/KirillVishnyakov/Architectural-Biases-in-Time-Series-Anomaly-Detection
!git -C /Architectural-Biases-in-Time-Series-Anomaly-Detection log --oneline -1

# path to the github repo
root_dir = "/root/Architectural-Biases-in-Time-Series-Anomaly-Detection"
sys.path.append(root_dir)

import utils.config as config
# arg1: path to the cats dataset, arg2: path to the checkpoint folder
config.init("/mnt/data/data/data.csv", root_dir + "/checkpoint_dir")

rm: cannot remove '/Architectural-Biases-in-Time-Series-Anomaly-Detection': No such file or directory
Cloning into 'Architectural-Biases-in-Time-Series-Anomaly-Detection'...
remote: Enumerating objects: 404, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 404 (delta 22), reused 34 (delta 12), pack-reused 356 (from 1)
Receiving objects: 100% (404/404), 100.61 MiB | 27.61 MiB/s, done.
Resolving deltas: 100% (208/208), done.
fatal: cannot change to '/Architectural-Biases-in-Time-Series-Anomaly-Detection': No such file or directory


### Setup the environment to have access to required repo functions

In [3]:
warnings.filterwarnings("ignore")
from models.transformer_encoder_forecaster import patch_transformer
import dataset
import train

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.__version__)
print(device)

2.8.0+cu129
cuda


### Random states for reproducibility

In [4]:
seed = 432
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [5]:
train_dataset_t = dataset.forecasting_Dataset(device = device, window_size = 256, horizon = 4, start = 0, end = 800000)
test_dataset_t = dataset.forecasting_Dataset(device = device, window_size = 256, horizon = 4, scaler = train_dataset_t.scaler, start = 800000, end = 1000000, train = False)

In [6]:
transformer_model = patch_transformer(lookback_window = 256, forecast_horizon = 4, d_model = 256, nhead = 8, dropout = 0.0, num_features = 17, num_blocks = 1).to(device)
transformer_model = torch.compile(transformer_model)
best_model_wts, best_loss, avg_train_loss = \
train.fit_forecaster(device, transformer_model, "transformer_encoder", train_dataset_t, test_dataset_t, lr = 0.001, batch_size = 512, num_epochs = 20)

32
LR Scheduler: 781 warmup steps, 31240 total steps
|transformer_encoder| train = 0.1849 | test= 0.0221 | LR: 9.98e-04
|transformer_encoder| train = 0.0220 | test= 0.0212 | LR: 9.86e-04
|transformer_encoder| train = 0.0215 | test= 0.0210 | LR: 9.60e-04
|transformer_encoder| train = 0.0213 | test= 0.0208 | LR: 9.23e-04
|transformer_encoder| train = 0.0212 | test= 0.0206 | LR: 8.76e-04
|transformer_encoder| train = 0.0211 | test= 0.0206 | LR: 8.18e-04
|transformer_encoder| train = 0.0210 | test= 0.0205 | LR: 7.53e-04
|transformer_encoder| train = 0.0209 | test= 0.0204 | LR: 6.81e-04
|transformer_encoder| train = 0.0208 | test= 0.0204 | LR: 6.04e-04
|transformer_encoder| train = 0.0209 | test= 0.0203 | LR: 5.25e-04
|transformer_encoder| train = 0.0207 | test= 0.0203 | LR: 4.45e-04
|transformer_encoder| train = 0.0207 | test= 0.0203 | LR: 3.67e-04
|transformer_encoder| train = 0.0207 | test= 0.0203 | LR: 2.93e-04
|transformer_encoder| train = 0.0206 | test= 0.0202 | LR: 2.24e-04
|transfor

In [7]:
torch.save(transformer_model._orig_mod.state_dict(), "tranformer_forecaster_weights.pt")